# सिनैप्टिक रूटिंग आर्किटेक्चर (एसआरए)
## 11: सिनैप्स का गतिशील विलोपन (हॉट-स्वैप को रद्द करना/विशिष्ट डोमेन को पर्ज करना)

इस नोटबुक में, हम SRA की एक विशेषता प्रदर्शित करते हैं: "सिनैप्टिक विलोपन।" विशेष रूप से, हम निम्नलिखित दो परिदृश्यों की जांच करेंगे।
1. **प्लगइन हटाएं (`pop_synapses`)**: बाद में जोड़े गए सिनैप्स को भौतिक रूप से हटाएं (हॉट-स्वैप) और उन्हें जोड़े जाने से पहले की स्थिति में पुनर्स्थापित करें।
2. ** विशिष्ट डोमेन को शुद्ध करें (`क्लियर_सिनैप्स`)**: प्रारंभिक आधार मॉडल से केवल विशिष्ट फ़ंक्शन (जैसे गणित) निकालें, और सुरक्षित रूप से "शून्य स्पष्ट" सिनैप्स निकालें जो दूसरों के साथ साझा नहीं किए जाते हैं।

*इस नोटबुक को बिना जीपीयू के चलाया जा सकता है।

In [ ]:
# 0. 環境セットアップ (Google Colab用)
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from sra_language_models import MoESRALanguageModel
from sra_experiment import find_unshared_synapses
import torch


### 1. प्लगइन्स हटाना (`pop_synapses`)
सबसे पहले, आइए एक छोटा मॉडल बनाएं, गतिशील रूप से सिनैप्स जोड़ें, और फिर उन्हें `pop_synapses` से हटा दें।

In [ ]:
# モデルの初期化
dim = 128
layers = 2
num_synapses = 4
k = 2
syn_hidden = 256
vocab_size = 1000

print("=== プラグインの取り外しデモ ===")
model = MoESRALanguageModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"[初期状態] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの追加 (Hot-Swap)
print("\nプラグインとしてシナプスを 2つ 追加します...")
model.add_synapses(2, freeze_base=True)
print(f"[追加後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの取り外し (Undo Hot-Swap)
print("\n追加したシナプスを 1つ 取り外します...")
model.pop_synapses(1)
print(f"[取り外し後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

इस तरह, `pop_synapses(N)` को कॉल करने से मॉडल की क्षमता बहाल करते हुए, अंत से N सिनैप्टिक टेंसर को भौतिक रूप से हटा दिया जाता है।

### 2. विशिष्ट डोमेन को शुद्ध करना (`clear_synapses` और `find_unshared_synapses`)
इसके बाद, हम बेस मॉडल के बीच से "अनावश्यक सिनैप्स जो केवल एक विशिष्ट डोमेन (इस मामले में, 'गणित') में उपयोग किए जाते हैं" का स्वचालित रूप से पता लगाने की प्रक्रिया का प्रदर्शन करेंगे, जिसने कई डोमेन सीखे हैं, और उन्हें शून्य-क्लियरिंग द्वारा सुरक्षित रूप से अक्षम कर दिया है।

In [ ]:
import random
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

device = "cuda" if torch.cuda.is_available() else "cpu"

# 仮のデータセットとバッチ関数の準備（デモ用）
domains = ["text", "code", "math"]
data = {d: torch.randint(0, vocab_size, (1000,)) for d in domains}

def dummy_get_batch(data_dict, batch_size, seq_len, domain):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    d_data = data_dict[domain]
    for i in range(batch_size):
        start = random.randint(0, len(d_data) - seq_len - 1)
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
    return x, y

# もう少し大きめのモデルを準備
multi_model = MoESRALanguageModel(vocab_size, dim=128, layers=2, num_synapses=16, k=2, syn_hidden=256).to(device)

In [ ]:
print("=== 特定ドメインのパージデモ ===")
print("Mathドメインでのみ使用され、他(Text, Code)と共用されていないシナプスを検索します...")

# ユーティリティを用いて対象シナプスを特定
unshared_synapses = find_unshared_synapses(
    model=multi_model, 
    data_dict=data, 
    target_domain="math", 
    other_domains=["text", "code"], 
    get_batch_func=dummy_get_batch,
    max_seq_len=32,
    threshold=0.01  # 1%以上の頻度で利用されていれば「使用している」と判定
)

print(f"\n抽出されたMath専用のシナプスインデックス: {unshared_synapses}")

*उपरोक्त सूचकांक अप्रशिक्षित मॉडल के मामले में नहीं मिल सकता है क्योंकि यह यादृच्छिक रूप से वितरित होता है।

संबंधित सिनैप्स को शून्य करने के लिए निकाले गए इंडेक्स (या यदि नहीं मिला तो डेमो प्रयोजनों के लिए उपयुक्त इंडेक्स) को `clear_synapses` में पास करें।

In [ ]:
if len(unshared_synapses) > 0:
    target_idx = unshared_synapses[0]
else:
    print("※未学習モデルのため条件に完全合致するものがありませんでした。デモとしてシナプス 3 を対象にします。")
    unshared_synapses = [3]
    target_idx = 3

# クリア前の重みのノルムを確認
pre_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
pre_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア前] シナプス {target_idx} の Router埋め込みノルム: {pre_emb_norm:.4f}, W1ノルム: {pre_w1_norm:.4f}")

# ゼロクリア実行
multi_model.clear_synapses(unshared_synapses)
print("\nゼロクリアを実行しました。\n")

# クリア後の重みのノルムを確認
post_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
post_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア後] シナプス {target_idx} の Router埋め込みノルム: {post_emb_norm:.4f}, W1ノルム: {post_w1_norm:.4f}")

### निष्कर्ष
`clear_synapses` निर्दिष्ट सूचकांक के भार को भौतिक रूप से हटाए बिना `0.0` तक साफ़ करता है।
यह अन्य सिनैप्स के इंडेक्स (आईडी) को बहने से रोकता है और मौजूदा मेटाडेटा मास्क के साथ संगतता बनाए रखते हुए अनावश्यक गणना पथों को पूरी तरह से अक्षम कर देता है। बाद में जब यह एक खाली स्लॉट बन जाता है तो एक नए प्लगइन के साथ इंडेक्स को ओवरराइट करना (हॉट-स्वैप) संभव है।